# LusoEnergia — Parte III: Prever o consumo do dia seguinte

**Problema preditivo:** prever o consumo total do **dia seguinte** (kWh/dia, agregação diária de Global_active_power).
Esta parte tem duas secções: 3.1 construção de variáveis candidatas (sem fuga de informação) e 3.2 seleção comparada (filter, wrapper, embedded) com validação temporal.

In [26]:
import sys, warnings
sys.path.insert(0, "../src")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from itertools import combinations

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LassoCV, LinearRegression, Lasso, Ridge
from sklearn.feature_selection import SelectKBest, RFE, SelectFromModel, mutual_info_regression
from sklearn.model_selection import TimeSeriesSplit, KFold, GridSearchCV, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.exceptions import ConvergenceWarning
from sklearn.impute import SimpleImputer
import holidays


warnings.filterwarnings("ignore", category=ConvergenceWarning)
pd.set_option("display.width", 1000)
pd.set_option("display.max_colwidth", None)

AZUL, LARANJA, TINTA, TINTA_2, GRELHA = "#2a78d6", "#e07b24", "#2b2b28", "#6b6a64", "#e4e3de"
CMAP = LinearSegmentedColormap.from_list("azul", ["#e9f1fc", "#9ec5f4", "#3987e5", "#184f95", "#0d366b"])
SEED = 42

def estilo(ax):
    for sp in ("top", "right"):
        ax.spines[sp].set_visible(False)
    for sp in ("left", "bottom"):
        ax.spines[sp].set_color(GRELHA)
    ax.tick_params(colors=TINTA_2, labelsize=9, length=0)
    ax.grid(axis="y", color=GRELHA, lw=0.8)
    ax.set_axisbelow(True)


# Carregamento dos dados

In [27]:
diario = pd.read_csv("data/processed/consumo_tratado.csv", index_col=0, parse_dates=True).asfreq("D")

# Temperatura horária externa (Sceaux) -- dado de enriquecimento (bónus)
tserie = (pd.read_csv("data/sceaux_temp_horaria_2008_2009.csv", parse_dates=["timestamp"])
          .set_index("timestamp")["temp_c"])
diario["temp"] = tserie.resample("D").mean().reindex(diario.index)


# Alinhar alvo y e matriz de features X_all
y = diario["Global_active_power"].dropna()
X = X_all.loc[y.index].copy()

# Configuração da validação temporal e número de features alvo
N_SPLITS = 5
K_FEAT = 10
tscv = TimeSeriesSplit(n_splits=N_SPLITS)

print(f"{diario['Global_active_power'].notna().sum()} dias válidos | {diario.index.min().date()} -> {diario.index.max().date()}")
diario.head()

547 dias válidos | 2008-05-01 -> 2009-10-31


,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3,imputado,temp
datetime,,,,,,,,,
2008-05-01,0.324,0.084,241.38,1.4,0.0,0.0,0.0,False,9.762500
2008-05-02,0.252,0.000,244.28,1.0,0.0,0.0,1.0,False,11.020833
2008-05-03,2.570,0.440,242.85,10.6,0.0,1.0,1.0,False,14.883333
2008-05-04,0.544,0.000,243.49,2.2,0.0,2.0,0.0,False,15.875000
2008-05-05,0.354,0.118,242.00,1.4,0.0,0.0,0.0,False,15.020833


## Criação das Variáveis Candidatas

Para esta parte procedeu-se à criação de 23 variáveis candidatas, sendo estas:

- lag_1, lag_2 e lag_7: que indicam os 3 lags apresentados no enunciado
- mm_3, mm_7 e mm_14: médias móveis dos últimos 3, 7 e 14 dias
- dp_7: desvio padrão móvel dos últimos 7 dias do consumo
- dif_lag1_lag7: diferença entre o consumo do dia anterior (lag_1) e o consumo do mesmo dia na semana passada (lag_7)
- fim_de_semana: binário que indica se é fim de semana ou não
- dow_sin e dow_cos: indica os dias da semana
- mes_sin e mes_cos: indica os meses do ano 
- feriado: binário que indica se é feriado ou não; Analisa-se de forma diferente por serem dias atípicos
- vespera_feriado: binário que indica se é ou não véspera de feriado; Analisa-se devido às noites anteriores aos feriados, os habitantes costumarem ficar mais tempo acordados
- lag_1 das submedições e do Global_reactive_power
- temp_lag1 e temp_mm7: lag e média móveis 


Existem variáveis que foram divididas em seno e cosseno devido, a evitar pensamento de hierarquia e também salto numérico entre o valor do início e do fim que não faz sentido.

Em seguida, fez-se um teste de fuga, para perceber se todas as variáveis candidatas respeitam a linha temporal e não conseguem ter acesso ao tempo que estamos a tentar prever




In [28]:
FERIADOS = holidays.country_holidays("FR", years=range(2008, 2011))   # Sceaux fica em França

def construir_features(d):
    # Variáveis para prever o dia t usando apenas informação até t-1 (e calendário de t)
    y = d["Global_active_power"]                        
    ys = y.shift(1)                                    # série "vista" ao fim do dia t-1
    X = pd.DataFrame(index=d.index)

    # --- Lags do consumo ---
    for k in (1, 2, 7):
        X[f"lag_{k}"] = y.shift(k)

    # --- Médias / dispersão móveis (janela termina em t-1) ---
    for w in (3, 7, 14):
        X[f"mm_{w}"] = ys.rolling(w, min_periods=int(np.ceil(0.7 * w))).mean()
    X["dp_7"] = ys.rolling(7, min_periods=5).std()
    X["dif_lag1_lag7"] = X["lag_1"] - X["lag_7"]       # variação face à mesma altura da semana anterior

    # --- Calendário (conhecido com antecedência, índice ao minuto) ---
    dow = d.index.day_of_week
    dia_norm = d.index.normalize()                        # ex.: 2008-05-01 00:00:00
    dia_seguinte = dia_norm + pd.Timedelta(days=1)

    # Ciclo diário (0..1439 minutos no dia; muito mais suave que só a hora)
    min_do_dia = d.index.hour * 60 + d.index.minute
    X["hora_sin"] = np.sin(2 * np.pi * min_do_dia / 1440)
    X["hora_cos"] = np.cos(2 * np.pi * min_do_dia / 1440)

    # Dia da semana e mês
    X["fim_semana"] = (dow >= 5).astype(int)
    X["dow_sin"] = np.sin(2 * np.pi * dow / 7)
    X["dow_cos"] = np.cos(2 * np.pi * dow / 7)
    X["mes_sin"] = np.sin(2 * np.pi * (d.index.month - 1) / 12)
    X["mes_cos"] = np.cos(2 * np.pi * (d.index.month - 1) / 12)

    # Feriados em França (comparação vetorizada por data)
    feriados_dates = set(FERIADOS.keys())
    X["feriado"] = pd.Series(dia_norm.date, index=d.index).isin(feriados_dates).astype(int)
    X["vespera_feriado"] = pd.Series(dia_seguinte.date, index=d.index).isin(feriados_dates).astype(int)

    # --- Outras grandezas do sistema, do dia anterior ---
    for c in ("Sub_metering_1", "Sub_metering_2", "Sub_metering_3", "Global_reactive_power"):
        X[f"{c}_lag1"] = d[c].shift(1)

    # --- Temperatura externa (observada até t-1) ---
    X["temp_lag1"] = d["temp"].shift(1)
    X["temp_mm7"] = d["temp"].shift(1).rolling(7, min_periods=5).mean()
    return X


X_all = construir_features(diario)
print(f"{X_all.shape[1]} variáveis candidatas")

23 variáveis candidatas


In [29]:
# Teste de fuga: corromper TUDO o que é medido do dia t em diante não pode alterar as variáveis até ao dia t.
def teste_fuga(d, n=25, seed=0):
    X0 = construir_features(d)
    rng = np.random.default_rng(seed)
    dias = rng.choice(d.index[20:-1], n, replace=False)
    cols_num = d.select_dtypes(include=[np.number]).columns

    for t in dias:
        d2 = d.copy()
        d2.loc[t:, cols_num] = d2.loc[t:, cols_num] + 1000
        X2 = construir_features(d2)
        assert X0.loc[:t].equals(X2.loc[:t]), f"FUGA detetada em {t.date()}"

    return f"OK: {n} perturbações aleatórias do futuro (a partir do dia t) não alteram as variáveis do dia t."

In [30]:
teste_fuga(diario, n=25, seed=0)

'OK: 25 perturbações aleatórias do futuro (a partir do dia t) não alteram as variáveis do dia t.'

In [31]:
FONTE = {
    "lag_1": "Global_active_power",
    "lag_2": "Global_active_power",
    "lag_7": "Global_active_power",
    "mm_3": "Global_active_power",
    "mm_7": "Global_active_power",
    "mm_14": "Global_active_power",
    "dp_7": "Global_active_power",
    "dif_lag1_lag7": "Global_active_power",
    "hora_sin": "datetime",
    "hora_cos": "datetime",
    "fim_semana": "datetime",
    "dow_sin": "datetime",
    "dow_cos": "datetime",
    "mes_sin": "datetime",
    "mes_cos": "datetime",
    "feriado": "datetime",
    "vespera_feriado": "datetime",
    "Sub_metering_1_lag1": "Sub_metering_1",
    "Sub_metering_2_lag1": "Sub_metering_2",
    "Sub_metering_3_lag1": "Sub_metering_3",
    "Global_reactive_power_lag1": "Global_reactive_power",
    "temp_lag1": "temp",
    "temp_mm7": "temp",
}

DESC = {
    "lag_1": "consumo de ontem",
    "lag_2": "consumo de anteontem",
    "lag_7": "consumo há 7 dias (mesmo dia da semana)",
    "mm_3": "média dos 3 dias anteriores",
    "mm_7": "média dos 7 dias anteriores",
    "mm_14": "média dos 14 dias anteriores",
    "dp_7": "desvio-padrão dos 7 dias anteriores",
    "dif_lag1_lag7": "lag_1 − lag_7",
    "hora_sin": "minuto do dia / ciclo de 24h (seno)",
    "hora_cos": "minuto do dia / ciclo de 24h (cosseno)",
    "fim_semana": "dia a prever é sábado/domingo",
    "dow_sin": "dia da semana (seno)",
    "dow_cos": "dia da semana (cosseno)",
    "mes_sin": "mês (seno)",
    "mes_cos": "mês (cosseno)",
    "feriado": "dia a prever é feriado (FR)",
    "vespera_feriado": "dia a prever é véspera de feriado",
    "Sub_metering_1_lag1": "cozinha, ontem (kWh)",
    "Sub_metering_2_lag1": "lavandaria + frigorífico, ontem (kWh)",
    "Sub_metering_3_lag1": "aquecedor de água + AC, ontem (kWh)",
    "Global_reactive_power_lag1": "energia reativa, ontem",
    "temp_lag1": "temperatura média de ontem",
    "temp_mm7": "temperatura média dos 7 dias anteriores",
}

assert set(FONTE) == set(X_all.columns) == set(DESC)

tab_feat = pd.DataFrame({"descrição": DESC, "fonte": FONTE})
tab_feat["usa dados até"] = [
    "t − 1" if f != "datetime" else "calendário de t (conhecido)"
    for f in tab_feat["fonte"]
]
tab_feat

,descrição,fonte,usa dados até
lag_1,consumo de ontem,Global_active_power,t − 1
lag_2,consumo de anteontem,Global_active_power,t − 1
lag_7,consumo há 7 dias (mesmo dia da semana),Global_active_power,t − 1
mm_3,média dos 3 dias anteriores,Global_active_power,t − 1
mm_7,média dos 7 dias anteriores,Global_active_power,t − 1
mm_14,média dos 14 dias anteriores,Global_active_power,t − 1
dp_7,desvio-padrão dos 7 dias anteriores,Global_active_power,t − 1
dif_lag1_lag7,lag_1 − lag_7,Global_active_power,t − 1
hora_sin,minuto do dia / ciclo de 24h (seno),datetime,calendário de t (conhecido)
hora_cos,minuto do dia / ciclo de 24h (cosseno),datetime,calendário de t (conhecido)


# Selecionar Variáveis

## Métodos utilizados

Para poder selecionar as variáveis que realmente vale apena utilizar usa-se 4 métodos:

- Baseline: treina a Regressão Linear com todas as variáveis, para perceber se uma variável é importante ou não
- Filter(SelectKBest + mutual_info_regression):  treina cada uma separadamente, para conseguirmos perceber quanta informação sobre o consumo cada variável tem
- Wrapper(RFE): treina o modelo com todas para descobrir a variável com menos peso e volta a treinar sem essa. Repete-se até k variáveis
- Embedded(LassoCV + SelectFromModel): por penalização das variáveis aos coeficientes, sendo descartadas as que têm coeficiente zero

In [42]:
def criar_pipelines(k_feat=10):
    prep = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])

    # 0. Baseline: todas as variáveis
    pipe_all = Pipeline([
        ("prep", clone(prep)),
        ("model", LinearRegression()),
    ])

    # 1. Filter: Informação Mútua
    pipe_filter = Pipeline([
        ("prep", clone(prep)),
        ("select", SelectKBest(
            score_func=lambda X_f, y_f: mutual_info_regression(X_f, y_f, random_state=42),
            k=k_feat
        )),
        ("model", LinearRegression()),
    ])

    # 2. Wrapper: RFE com Ridge
    pipe_rfe = Pipeline([
        ("prep", clone(prep)),
        ("select", RFE(estimator=Ridge(alpha=1.0), n_features_to_select=k_feat, step=1)),
        ("model", LinearRegression()),
    ])

    # 3. Embedded: LassoCV com split temporal interno (sem baralhar!)
    lasso_cv = LassoCV(cv=TimeSeriesSplit(n_splits=3), random_state=42, max_iter=10000)
    pipe_lasso = Pipeline([
        ("prep", clone(prep)),
        ("select", SelectFromModel(lasso_cv, threshold=1e-5, max_features=k_feat)),
        ("model", LinearRegression()),
    ])

    return {
        "Todas (Baseline)": pipe_all,
        "Filter (Mutual Info)": pipe_filter,
        "Wrapper (RFE)": pipe_rfe,
        "Embedded (Lasso)": pipe_lasso,
    }

## Função de avaliação temporal e cálculo de estabilidade

Foi calculado o Erro Médio Absoluto(MAE) e a Raiz do Erro Quadrático Médio (RMSE) para perceber qual dos métodos têm melhores resultados. Observou-se que todos os métodos foram melhores que o Baseline, isto deve-se ao problema da multicolinearidade entre as variáveis, por isto motivo esta é descartada.

O método com melhor predição é o Lasso, contudo como este tem uma baixa estabilidade entre folds decidiu-se avançar com os três métodos para verificar quantas vezes cada variável foi escolhida por cada um deles.

In [56]:
def avaliar_selecao_temporal(pipes, X, y, cv):
    resultados = []
    selecoes_por_metodo = {}
    nomes_cols = np.array(X.columns)

    for nome, pipe in pipes.items():
        maes, rmses, folds_cols = [], [], []

        for train_idx, val_idx in cv.split(X):
            X_tr, y_tr = X.iloc[train_idx], y.iloc[train_idx]
            X_va, y_va = X.iloc[val_idx], y.iloc[val_idx]

            # Remove linhas onde y_tr seja NaN (segurança extra)
            valid_tr = y_tr.notna()
            X_tr, y_tr = X_tr.loc[valid_tr], y_tr.loc[valid_tr]

            valid_va = y_va.notna()
            X_va, y_va = X_va.loc[valid_va], y_va.loc[valid_va]

            prep = clone(pipe.named_steps["prep"])
            modelo = clone(pipe.named_steps["model"])

            X_tr_p = prep.fit_transform(X_tr)
            X_va_p = prep.transform(X_va)

            if "select" in pipe.named_steps:
                seletor = clone(pipe.named_steps["select"])
                seletor.fit(X_tr_p, y_tr)
                mask = seletor.get_support()

                if not np.any(mask) or seletor.transform(X_tr_p).shape[1] == 0:
                    # Fallback: escolhe a coluna com maior correlação absoluta com y_tr
                    corrs = []
                    for j in range(X_tr_p.shape[1]):
                        c = np.corrcoef(X_tr_p[:, j], y_tr.to_numpy())[0, 1]
                        corrs.append(0.0 if np.isnan(c) else abs(c))
                    idx_best = int(np.argmax(corrs))
                    mask = np.zeros(X_tr_p.shape[1], dtype=bool)
                    mask[idx_best] = True

                X_tr_sel = X_tr_p[:, mask]
                X_va_sel = X_va_p[:, mask]
                cols_fold = set(nomes_cols[mask])
            else:
                X_tr_sel = X_tr_p
                X_va_sel = X_va_p
                cols_fold = set(nomes_cols)

            modelo.fit(X_tr_sel, y_tr)
            pred = modelo.predict(X_va_sel)

            maes.append(mean_absolute_error(y_va, pred))
            rmses.append(np.sqrt(mean_squared_error(y_va, pred)))
            folds_cols.append(cols_fold)

        jaccards = [
            len(folds_cols[i] & folds_cols[j]) / len(folds_cols[i] | folds_cols[j])
            for i in range(len(folds_cols))
            for j in range(i + 1, len(folds_cols))
        ]

        selecoes_por_metodo[nome] = folds_cols
        resultados.append({
            "Método": nome,
            "MAE médio": round(np.mean(maes), 4),
            "RMSE médio": round(np.mean(rmses), 4),
            "Estabilidade (Jaccard)": round(np.mean(jaccards), 3) if jaccards else 1.0,
        })

    return pd.DataFrame(resultados), selecoes_por_metodo

df_resultados, selecoes = avaliar_selecao_temporal(pipelines, X, y, tscv)
df_resultados

C:\Users\adria\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\numpy\lib\_function_base_impl.py:3037: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
C:\Users\adria\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\numpy\lib\_function_base_impl.py:3038: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


,Método,MAE médio,RMSE médio,Estabilidade (Jaccard)
0,Todas (Baseline),0.5680,0.9223,1.000
1,Filter (Mutual Info),0.5125,0.8365,0.482
2,Wrapper (RFE),0.5645,0.8850,0.470
3,Embedded (Lasso),0.4464,0.7882,0.286


## Tabela de estabilidade por variável entre folds
É visto durante os 5 folds feitos quantas vezes cada variável foi selecionada, o máximo é 15.

In [51]:
freq_selecao = pd.DataFrame(index=X.columns)

for nome in ["Filter (Mutual Info)", "Wrapper (RFE)", "Embedded (Lasso)"]:
    freq_selecao[nome] = [
        sum(col in s for s in selecoes[nome])
        for col in X.columns
    ]

freq_selecao["Total (máx 15)"] = freq_selecao.sum(axis=1)
freq_selecao["Fonte"] = freq_selecao.index.map(FONTE)

# Ordenar das mais escolhidas para as menos escolhidas
freq_selecao = freq_selecao.sort_values(by=["Total (máx 15)", "Wrapper (RFE)"], ascending=False)
freq_selecao

,Filter (Mutual Info),Wrapper (RFE),Embedded (Lasso),Total (máx 15),Fonte
fim_semana,2,5,5,12,datetime
mes_cos,4,5,3,12,datetime
dp_7,4,5,2,11,Global_active_power
mm_3,4,3,3,10,Global_active_power
Global_reactive_power_lag1,4,3,2,9,Global_reactive_power
mm_7,3,4,1,8,Global_active_power
temp_lag1,5,3,0,8,temp
dow_cos,0,4,3,7,datetime
temp_mm7,3,4,0,7,temp
dif_lag1_lag7,3,3,1,7,Global_active_power


## Comparação dos subconjuntos finais e interseção entre métodos

Para escolher as variáveis, definiu-se que mínimo de vezes escolhida 3 por cada método, nos 5 folds efetuados. Escolheu-se as variáveis que cumprissem isso em 2 métodos.



In [57]:
limiar_folds = 3

sub_filter = set(freq_selecao[freq_selecao["Filter (Mutual Info)"] >= limiar_folds].index)
sub_rfe = set(freq_selecao[freq_selecao["Wrapper (RFE)"] >= limiar_folds].index)
sub_lasso = set(freq_selecao[freq_selecao["Embedded (Lasso)"] >= limiar_folds].index)

consenso_3 = sub_filter & sub_rfe & sub_lasso
consenso_2_de_3 = {
    col for col in freq_selecao.index
    if (col in sub_filter) + (col in sub_rfe) + (col in sub_lasso) >= 2
}

print("Consenso nos 3 métodos:", sorted(consenso_3))
print("Presentes em >= 2 dos 3 métodos:", sorted(consenso_2_de_3))

Consenso nos 3 métodos: ['mes_cos', 'mm_3']
Presentes em >= 2 dos 3 métodos: ['Global_reactive_power_lag1', 'dif_lag1_lag7', 'dow_cos', 'dp_7', 'fim_semana', 'mes_cos', 'mm_3', 'mm_7', 'temp_lag1', 'temp_mm7']


In [62]:
# 1. Definir o subconjunto final (consenso em >= 2 dos 3 métodos, com >= 3 folds)
cols_finais = sorted(consenso_2_de_3)

# Tabela justificativa das variáveis escolhidas
justificacao_df = pd.DataFrame({
    "variavel": cols_finais,
    "fonte": [FONTE[c] for c in cols_finais],
    "descricao": [DESC[c] for c in cols_finais],
    "filter_mi": [freq_selecao.loc[c, "Filter (Mutual Info)"] for c in cols_finais],
    "wrapper_rfe": [freq_selecao.loc[c, "Wrapper (RFE)"] for c in cols_finais],
    "embedded_lasso": [freq_selecao.loc[c, "Embedded (Lasso)"] for c in cols_finais],
    "total_15": [freq_selecao.loc[c, "Total (máx 15)"] for c in cols_finais],
}).sort_values(by="total_15", ascending=False).reset_index(drop=True)

display(justificacao_df)

# 2. Construir o dataset final apenas com as variáveis selecionadas
X_sel = X[cols_finais].copy()

print(f"Variáveis candidatas avaliadas: {X.shape[1]} -> Variáveis selecionadas: {X_sel.shape[1]}")
print(f"Dimensão final de X_sel: {X_sel.shape}")

# 3. Guardar dataset com as variáveis selecionadas
X_sel.to_csv("data/vars_candidatas/consumption_varselect.csv", index=True)

,variavel,fonte,descricao,filter_mi,wrapper_rfe,embedded_lasso,total_15
0,fim_semana,datetime,dia a prever é sábado/domingo,2,5,5,12
1,mes_cos,datetime,mês (cosseno),4,5,3,12
2,dp_7,Global_active_power,desvio-padrão dos 7 dias anteriores,4,5,2,11
3,mm_3,Global_active_power,média dos 3 dias anteriores,4,3,3,10
4,Global_reactive_power_lag1,Global_reactive_power,"energia reativa, ontem",4,3,2,9
5,temp_lag1,temp,temperatura média de ontem,5,3,0,8
6,mm_7,Global_active_power,média dos 7 dias anteriores,3,4,1,8
7,dif_lag1_lag7,Global_active_power,lag_1 − lag_7,3,3,1,7
8,dow_cos,datetime,dia da semana (cosseno),0,4,3,7
9,temp_mm7,temp,temperatura média dos 7 dias anteriores,3,4,0,7


Variáveis candidatas avaliadas: 23 -> Variáveis selecionadas: 10
Dimensão final de X_sel: (547, 10)


## Verificar das variáveis selecionadas, quantas provém de cada variável do dataset original

Isto serve para verificar que variáveis são mais importantes para prever o consumo do cliente. Serve para que novas análises sejam mais económicas mas não menos precisas.

As variáveis que consideramos não necessárias são as submedições.

In [58]:
resumo_fontes = (
    freq_selecao.groupby("Fonte")[["Filter (Mutual Info)", "Wrapper (RFE)", "Embedded (Lasso)", "Total (máx 15)"]]
    .agg(["count", "mean", "max"])
)
resumo_fontes

Filter (Mutual Info)               Wrapper (RFE)               Embedded (Lasso)               Total (máx 15)              
                                     count      mean max         count      mean max            count      mean max          count      mean max
Fonte                                                                                                                                           
Global_active_power                      8  3.000000   4             8  2.375000   5                8  1.000000   3              8  6.375000  11
Global_reactive_power                    1  4.000000   4             1  3.000000   3                1  2.000000   2              1  9.000000   9
Sub_metering_1                           1  1.000000   1             1  1.000000   1                1  1.000000   1              1  3.000000   3
Sub_metering_2                           1  0.000000   0             1  1.000000   1                1  3.000000   3              1  4.000000   4
Sub_metering_3                           1  0.000000   0             1  0.000000   0                1  0.000000   0              1  0.000000   0
datetime                                 9  1.444444   4             9  2.111111   5                9  1.777778   5              9  5.333333  12
temp                                     2  4.000000   5             2  3.500000   4                2  0.000000   0              2  7.500000   8